In [36]:
#devaraj

In [37]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

from tensorflow.keras.utils import to_categorical

In [38]:
df = pd.read_csv("test.csv")

print(df.head())
print(df.info())

   Class Index                                              Title  \
0            3                  Fears for T N pension after talks   
1            4  The Race is On: Second Private Team Sets Launc...   
2            4      Ky. Company Wins Grant to Study Peptides (AP)   
3            4      Prediction Unit Helps Forecast Wildfires (AP)   
4            4        Calif. Aims to Limit Farm-Related Smog (AP)   

                                         Description  
0  Unions representing workers at Turner   Newall...  
1  SPACE.com - TORONTO, Canada -- A second\team o...  
2  AP - A company founded by a chemistry research...  
3  AP - It's barely dawn when Mike Fitzpatrick st...  
4  AP - Southern California's smog-fighting agenc...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7600 entries, 0 to 7599
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Class Index  7600 non-null   int64 
 1   Title        7600 non

In [16]:
df["text"] = df["Title"] + " " + df["Description"]

print(df["text"].head())

0    Fears for T N pension after talks Unions repre...
1    The Race is On: Second Private Team Sets Launc...
2    Ky. Company Wins Grant to Study Peptides (AP) ...
3    Prediction Unit Helps Forecast Wildfires (AP) ...
4    Calif. Aims to Limit Farm-Related Smog (AP) AP...
Name: text, dtype: object


In [17]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df["text"] = df["text"].apply(clean_text)

print(df["text"].head())

0    fears for t n pension after talks unions repre...
1    the race is on second private team sets launch...
2    ky company wins grant to study peptides ap ap ...
3    prediction unit helps forecast wildfires ap ap...
4    calif aims to limit farmrelated smog ap ap sou...
Name: text, dtype: object


In [18]:
X = df["text"]

# Convert class labels from 1-4 to 0-3
y = df["Class Index"] - 1

print(X.head())
print(y.head())

0    fears for t n pension after talks unions repre...
1    the race is on second private team sets launch...
2    ky company wins grant to study peptides ap ap ...
3    prediction unit helps forecast wildfires ap ap...
4    calif aims to limit farmrelated smog ap ap sou...
Name: text, dtype: object
0    2
1    3
2    3
3    3
4    3
Name: Class Index, dtype: int64


In [19]:
max_words = 10000

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)

max_length = 100

X_pad = pad_sequences(
    X_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print(X_pad.shape)

(7600, 100)


In [20]:
y = to_categorical(y, num_classes=4)

X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data :", X_train.shape)
print("Testing Data  :", X_test.shape)

Training Data : (6080, 100)
Testing Data  : (1520, 100)


In [26]:
model = Sequential()

model.add(Embedding(
    input_dim=max_words,
    output_dim=128
))

model.add(SimpleRNN(64))

model.add(Dense(32, activation="relu"))

model.add(Dense(4, activation="softmax"))

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_2 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)

Epoch 1/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.2368 - loss: 1.3968 - val_accuracy: 0.2623 - val_loss: 1.3941
Epoch 2/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.2488 - loss: 1.3898 - val_accuracy: 0.2516 - val_loss: 1.3896
Epoch 3/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.2613 - loss: 1.3884 - val_accuracy: 0.2516 - val_loss: 1.3877
Epoch 4/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.2525 - loss: 1.3916 - val_accuracy: 0.2574 - val_loss: 1.3875
Epoch 5/5
152/152 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.2599 - loss: 1.3994 - val_accuracy: 0.2730 - val_loss: 1.3856


In [28]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2612 - loss: 1.3875
Test Accuracy: 0.2611842155456543


In [31]:
model.save("ag_news_rnn.keras")

In [32]:
news = "Apple launches a new AI-powered smartphone."

news = clean_text(news)

seq = tokenizer.texts_to_sequences([news])

pad = pad_sequences(
    seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

prediction = model.predict(pad)

classes = [
    "World",
    "Sports",
    "Business",
    "Science/Technology"
]

print("Predicted Class:", classes[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Predicted Class: World
